In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Layer
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 데이터 준비
# 데이터 생성
X, y = make_regression(n_samples=1000, n_features=10, noise=0.1)
y = y.reshape(-1, 1)  # 출력 형태를 맞추기 위해 reshape

# 데이터 분리 및 정규화
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)
y_train = scaler_y.fit_transform(y_train)
y_test = scaler_y.transform(y_test)



In [2]:
# 2. 사용자 정의 층
class MyDenseLayer(Layer):
    def __init__(self, units, **kwargs):
        super(MyDenseLayer, self).__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        self.weight = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer="random_normal",
            trainable=True,
            name="weight"
        )
        self.bias = self.add_weight(
            shape=(self.units,),
            initializer="zeros",
            trainable=True,
            name="bias"
        )

    def call(self, inputs):
        z = tf.matmul(inputs, self.weight) + self.bias
        return tf.nn.relu(z)



In [3]:
# 3. 사용자 정의 손실 함수
def huber_loss(y_true, y_pred, delta=1.0):
    error = y_true - y_pred
    is_small_error = tf.abs(error) <= delta
    squared_loss = 0.5 * tf.square(error)
    linear_loss = delta * (tf.abs(error) - 0.5 * delta)
    return tf.where(is_small_error, squared_loss, linear_loss)



In [4]:
# 4. 모델 설계 및 훈련
# 사용자 정의 층을 활용한 신경망 모델
class MyModel(Model):
    def __init__(self):
        super(MyModel, self).__init__()
        self.hidden1 = MyDenseLayer(32, name="hidden1")
        self.hidden2 = MyDenseLayer(32, name="hidden2")
        self.output_layer = MyDenseLayer(1, name="output")

    def call(self, inputs):
        x = self.hidden1(inputs)
        x = self.hidden2(x)
        return self.output_layer(x)

# 모델 생성
model = MyModel()

# 모델 컴파일
model.compile(optimizer=Adam(), loss=huber_loss, metrics=["mse"])

# 모델 학습
history = model.fit(X_train, y_train, batch_size=32, epochs=10, validation_split=0.2)



Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.4170 - mse: 0.9933 - val_loss: 0.3805 - val_mse: 0.9132
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4079 - mse: 0.9708 - val_loss: 0.3512 - val_mse: 0.8334
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3540 - mse: 0.8023 - val_loss: 0.2831 - val_mse: 0.6644
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2845 - mse: 0.6529 - val_loss: 0.2266 - val_mse: 0.5471
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2089 - mse: 0.4940 - val_loss: 0.2187 - val_mse: 0.5313
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2062 - mse: 0.4799 - val_loss: 0.2185 - val_mse: 0.5308
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2208 - mse: 0.5185 - val_loss: 0.2183 - val_mse: 0.5305
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2111 - mse: 0.5106 - val_loss: 0.2183 - val_mse: 0.5304
Epoch 9/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2153 - mse: 

In [5]:
# 5. 평가 및 예측
# 테스트 데이터에서의 MSE
test_mse = model.evaluate(X_test, y_test, verbose=0)[1]
print(f"테스트 데이터에서의 MSE: {test_mse:.4f}")

# 첫 번째 샘플의 예측값과 실제값
y_pred = model.predict(X_test[:1])
y_pred_original = scaler_y.inverse_transform(y_pred)
y_test_original = scaler_y.inverse_transform(y_test[:1])
print(f"첫 번째 샘플의 예측값: {y_pred_original[0][0]:.4f}")
print(f"첫 번째 샘플의 실제값: {y_test_original[0][0]:.4f}")


테스트 데이터에서의 MSE: 0.5472
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
첫 번째 샘플의 예측값: 241.6831
첫 번째 샘플의 실제값: 235.9375
